In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd

df = pd.read_csv(
    '/kaggle/input/datasets/kazanova/sentiment140/training.1600000.processed.noemoticon.csv',
    encoding='ISO-8859-1',
    header=None
)

In [ ]:
df.head()

In [ ]:
df.columns = [
    "target",
    "id",
    "date",
    "flag",
    "user",
    "text"
]

In [ ]:
df.sample(5)

In [ ]:
df.shape

In [ ]:
df = df.drop(columns = [ "id",
    "date",
    "flag",
    "user"])

In [ ]:
df.head()

In [ ]:
print(df.isnull().sum())
print(df.duplicated().sum())

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.sample(10)

In [ ]:
df.shape

In [ ]:
import re 

def cleaning(text) : 

    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
df['clean_text'] = df['text'].apply(cleaning)

In [ ]:
df.sample(10)

In [ ]:
df = df.drop(columns = ['text'])

In [ ]:
df.sample(5)

In [ ]:
X = df['clean_text']
y = df['target']

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
nltk.download("punkt")
nltk.download("punkt_tab")


In [ ]:
df['token'] = df['clean_text'].apply(word_tokenize)

In [ ]:
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

In [ ]:
def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

In [ ]:
df['token'] = df['token'].apply(remove_stopwords)

In [ ]:
df['token']

In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

In [ ]:
def stem_words(tokens):
    return [stemmer.stem(word) for word in tokens]

In [ ]:
df['token'] = df['token'].apply(stem_words)

In [ ]:
df['token']

In [ ]:
df["processed_text"] = df["token"].apply(
    lambda token: " ".join(token)
)

In [ ]:
df.head()

In [ ]:
df.sample(5)

In [ ]:
X = df['processed_text']
y = df['target']

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state = 42,stratify = y)

In [ ]:
X_train.shape

In [ ]:
y_train.shape

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range = (1,2),
                            min_df = 2)

In [ ]:
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

In [ ]:
model.fit(X_train,y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

print(accuracy_score(y_test,y_pred))

In [ ]:
print(confusion_matrix(y_pred,y_test))

In [ ]:
classification_report(y_pred,y_test)

In [ ]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences = df['token'],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=0
)

In [ ]:
len(model.wv)

In [ ]:
model.wv['love']

In [ ]:
model.wv['love'].shape

In [ ]:
model.wv.most_similar("love", topn=5)

In [ ]:
import numpy as np

def document_vector(tokens):
    vectors = [
        model.wv[word]
        for word in tokens
        if word in model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [ ]:
X_w2v_text = df["token"]
y = df["target"]

X_train_w2v_text, X_test_w2v_text, y_train_w2v, y_test_w2v = train_test_split(
    X_w2v_text,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print(X_train_w2v_text.iloc[0])

In [ ]:
vec = document_vector(X_train_w2v_text.iloc[0])

print(vec)
print(vec.shape)

In [ ]:
X_train_w2v = np.array(
    X_train_w2v_text.apply(document_vector).tolist()
)

X_test_w2v = np.array(
    X_test_w2v_text.apply(document_vector).tolist()
)

In [ ]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(max_iter=1000)

model_lr.fit(X_train_w2v, y_train_w2v)

In [ ]:
y_pred_w2v = model_lr.predict(X_test_w2v)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_w2v = accuracy_score(y_test_w2v, y_pred_w2v)

print("Word2Vec Accuracy:", accuracy_w2v)

In [ ]:
from sklearn.metrics import confusion_matrix

cm_w2v = confusion_matrix(y_test_w2v, y_pred_w2v)

print(cm_w2v)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test_w2v, y_pred_w2v))